# Karachi AQI — Exploratory Data Analysis

Quick EDA on the backfilled feature data. Run after `backfill.py` has generated `data/backfill.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True

df = pd.read_csv('../data/backfill.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print(f'Shape: {df.shape}')
print(f'Date range: {df.timestamp.min()} → {df.timestamp.max()}')
df.head(3)

In [ ]:
# Missing value summary
miss = df.isnull().mean().sort_values(ascending=False)
print('Missing fraction per column:')
print(miss[miss > 0].to_string())

In [ ]:
# AQI over time
fig, ax = plt.subplots()
ax.plot(df['timestamp'], df['aqi_us'], linewidth=0.8, color='steelblue', alpha=0.85)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30)
ax.set_title('US AQI — Karachi (historical)')
ax.set_ylabel('US AQI')
plt.tight_layout()
plt.savefig('../report/eda_aqi_timeseries.png', dpi=150)
plt.show()

In [ ]:
# Hourly pattern
hourly_mean = df.groupby('hour')['aqi_us'].mean()
fig, ax = plt.subplots()
ax.bar(hourly_mean.index, hourly_mean.values, color='coral')
ax.set_title('Mean US AQI by Hour of Day')
ax.set_xlabel('Hour (local)')
ax.set_ylabel('Mean AQI')
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.savefig('../report/eda_hourly_pattern.png', dpi=150)
plt.show()

In [ ]:
# Correlation matrix
num_cols = ['pm2_5', 'pm10', 'no2', 'o3', 'aqi_us',
            'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right')
ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Correlation Matrix — AQI Features')
plt.tight_layout()
plt.savefig('../report/eda_correlation.png', dpi=150)
plt.show()

In [ ]:
# AQI distribution
fig, ax = plt.subplots()
df['aqi_us'].hist(bins=40, color='steelblue', alpha=0.8, ax=ax)
for threshold, label in [(50,'Good'), (100,'Moderate'), (150,'Sensitive'), (200,'Unhealthy')]:
    ax.axvline(threshold, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.text(threshold+2, ax.get_ylim()[1]*0.9, label, fontsize=7, color='red')
ax.set_title('US AQI Distribution — Karachi')
ax.set_xlabel('US AQI')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../report/eda_aqi_distribution.png', dpi=150)
plt.show()

In [ ]:
# Summary stats
print(df[['pm2_5','pm10','no2','o3','aqi_us','temperature_2m',
          'relative_humidity_2m','wind_speed_10m']].describe().round(2).to_string())